# score_0.94402 노트북 개요
이 노트북은 모든 문제를 다시 푸는 학습 노트북이 아니라, **기존 제출 결과가 서로 갈린 샘플만 추려서 Thinking 모델로 재판정**하는 후처리 실험입니다.
즉, 기본 submission을 만든 뒤 애매한 문제만 별도 모델로 덮어써 최종 점수를 올리는 흐름입니다.

## 실행 환경 준비
Unsloth, transformers, wandb 등 Thinking 추론에 필요한 패키지를 Colab/로컬 환경에 맞게 설치합니다.
이 노트북은 학습보다 추론과 결과 병합이 핵심이라 환경 셋업이 먼저 나옵니다.

In [1]:
# colab & local PC용 라이브러리 설치
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    import torch; v = re.match(r"[0-9\.]{3,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.32.post2" if v == "2.8.0" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1,<4.0.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth
!pip install transformers==4.57.0
!pip install --no-deps trl==0.22.2
!pip install wandb scikit-learn albumentations # 추가 라이브러리
!pip install ninja
# Blackwell 전용, CPU 풀가동 빌드
# 1. 빌드된 결과물을 휠 파일로 저장 (최초 1회)
# !MAX_JOBS=$(nproc) TORCH_CUDA_ARCH_LIST="10.0" pip wheel flash-attn --no-build-isolation -v -w /content/drive/MyDrive/my_wheels/
# 2. 다음 세션에서 설치할 때 (빛의 속도)
# !pip install /content/drive/MyDrive/my_wheels/flash_attn-*.whl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.4/43.4 MB 43.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 40.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 630.8/630.8 kB 81.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 415.2/415.2 kB 60.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 84.6 MB/s eta 0:00:00
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
unsloth-zoo 2026.4.2 requires msgspec, which is not installed.
unsloth-zoo 2026.4.2 requires tyro, which is not installed.
trl 1.0.0 requires datasets>=4.7.0, but you have datasets 3.6.0 

# 1. Thinking 모델 사용

## 1.1. 데이터 분류

Test Set 에서 이진 분류 수행
1. 쉬운 문제
2. 모호한/어려운 문제

**분류 기준**
1. 지금까지 제출한 submission.csv 파일을 모두 가져와서, 모두 같은 정답을 고른 경우 => 쉬운 문제
2. 모든 submission.csv 에서 하나라도 다른 정답을 고른 공우 => 모호한/어려운 문제

## 1.2. Thinking 모델로 추론

모호한/어려운 문제에 대해서 Qwen3-VL-32B-Thinking 모델로 추론 수행

## 추론용 공통 import
이미지 로딩, 데이터프레임 처리, Unsloth 모델 호출에 필요한 공통 모듈을 불러옵니다.
뒤 셀에서 prompt 생성과 배치 추론에 바로 사용됩니다.

In [1]:
from torch.utils.data import Dataset, DataLoader
from dataclasses import dataclass
from typing import Dict, List, Any
import pandas as pd
from PIL import Image
from transformers import (
    AutoModelForVision2Seq,
    AutoProcessor,
    BitsAndBytesConfig,
    get_linear_schedule_with_warmup
)
from tqdm.auto import tqdm
from unsloth import FastVisionModel # FastLanguageModel for LLMs
import torch

# 디바이스 GPU 우선 사용 설정
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

/tmp/ipykernel_19849/815104415.py:13: UserWarning: WARNING: Unsloth should be imported before [transformers] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastVisionModel # FastLanguageModel for LLMs


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Device: cuda


## 데이터/로그 경로 연결
Google Drive를 마운트하고 wandb를 로그인해 입력 CSV와 결과 파일을 읽고 저장할 준비를 합니다.

In [2]:
# 구글드라이브 마운트
from google.colab import drive, userdata
import wandb
import os

drive.mount('/content/drive')
%cd '/content/drive/MyDrive/Colab Notebooks/AI_2_ssafy/__AI챌린지'


# Colab Secrets에서 키를 가져와 환경 변수로 설정
os.environ["WANDB_API_KEY"] = userdata.get('WANDB_KEY')

# 자동으로 환경 변수를 읽어 로그인 시도
wandb.login()

Mounted at /content/drive
/content/drive/MyDrive/Colab Notebooks/AI_2_ssafy/__AI챌린지


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: qaz000219 (qaz000219-seoul-national-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## 기존 제출 파일 로드
여기서는 test.csv와 여러 submission 파일을 함께 읽습니다.
핵심 아이디어는 여러 제출이 **서로 같은 답을 낸 문제는 그대로 두고**, 답이 갈린 문제만 다시 검토하는 것입니다.

In [3]:
import pandas as pd

df_test = pd.read_csv('test.csv')

sub1 = pd.read_csv('sub1.csv')
sub2 = pd.read_csv('sub2.csv')
sub3 = pd.read_csv('sub3.csv')
sub4 = pd.read_csv('sub4.csv')
sub5 = pd.read_csv('sub5.csv')


'''
submission.csv
Complete · C044_정지성_1514758 · 3h ago
0.94402
'''
main_sub = pd.read_csv('main_submission.csv')

## 쉬운 문제 / 어려운 문제 분리
여러 submission의 answer 열을 가로로 붙인 뒤, 문제별 합의도를 계산합니다.
이 결과로 Thinking 모델을 돌릴 대상을 `ids`로 추립니다.

In [8]:
# 1. 서브미션 리스트 정의 (반드시 모든 sub 파일이 로드되어 있어야 함)
subs = [sub1, sub2, sub3, sub4, sub5]

# 2. answer 컬럼들을 하나의 데이터프레임으로 통합 (가로 방향)
combined_answers = pd.concat([s['answer'] for s in subs], axis=1)

# 3. 행(row)별로 가장 많이 등장한 값의 빈도수가 3 이상인지 확인
# value_counts().max()는 해당 행에서 가장 흔한 답변의 개수를 반환함
mask = combined_answers.apply(lambda x: x.value_counts().max() >= 4, axis=1)

In [9]:
mask.value_counts()

,count
True,4848
False,226


In [10]:
ids = set(df_test[mask == False].id)
print(f"어려운/모호한 문제의 개수: {len(ids)}")

어려운/모호한 문제의 개수: 226


## Thinking 모델 로드
애매한 샘플만 다시 보도록 Qwen3-VL Thinking 모델을 4bit로 불러옵니다.
전체 test를 재추론하는 것이 아니라 선별된 문제 집합에만 쓰는 점이 이 노트북의 핵심입니다.

In [11]:
model, tokenizer = FastVisionModel.from_pretrained(
    "unsloth/Qwen3-VL-32B-Thinking-unsloth-bnb-4bit",
    load_in_4bit = True, # Use 4bit to reduce memory use. False for 16bit LoRA.
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for long context
)

==((====))==  Unsloth 2026.4.2: Fast Qwen3_Vl patching. Transformers: 4.57.0.
   \\   /|    NVIDIA RTX PRO 6000 Blackwell Server Edition. Num GPUs = 1. Max memory: 94.971 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

model-00003-of-00006.safetensors:   0%|          | 0.00/4.79G [00:00<?, ?B/s]

model-00001-of-00006.safetensors:   0%|          | 0.00/4.79G [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/817 [00:00<?, ?B/s]

## Thinking 프롬프트 구성
이 셀은 재활용 VQA 도메인에 맞춘 시스템 지시문과 객관식 프롬프트 생성 함수를 정의합니다.
정답 선택뿐 아니라 근거를 생각한 뒤 마지막에 선택지를 내도록 유도하는 구조입니다.

In [12]:
# ===== 시스템 프롬프트 (Bilingual: English & Korean) =====
SYSTEM_INSTRUCT = (
    "Role: You are an expert AI specialized in recycling image analysis and VQA. "
    "당신은 재활용 관련 이미지를 분석하고 재활용품, 용기, 폐기물에 관한 객관식 질문에 답하는 전문가입니다.\n\n"

    "CORE SKILLS / 핵심 역량:\n"
    "1. Precision Counting: Count each item individually without skipping. (객체 개수 정밀 파악)\n"
    "2. Material Identification: Distinguish plastic, glass, metal, paper, and vinyl by texture/appearance. (재질 식별)\n"
    "3. Spatial & Visual Recognition: Identify colors, shapes of containers, and labels. (색상 및 형태 인식)\n"
    "4. K-Recycle Standards: Apply standard Korean recycling categories and classifications. (한국 재활용 분류 체계 적용)\n\n"

    "OUTPUT FORMAT (STRICT): / 출력 형식 (엄격 준수)\n"
    "- Respond with exactly ONE lowercase letter: a, b, c, or d\n"
    "- No explanation, punctuation, spaces, or line breaks. JUST THE LETTER.\n"
    "- 오직 소문자 a, b, c, d 중 하나만 답변하십시오. 설명, 구두점, 공백은 일절 금지합니다.\n\n"

    "ANALYSIS PROTOCOL / 분석 프로토콜:\n"
    "1. Full Image Scan: Analyze from top to bottom (portrait-oriented). (전체 영역 스캔)\n"
    "2. Quantitative VQA: For counting, verify each instance carefully. (개수 질문: 누락 없이 확인)\n"
    "3. Qualitative VQA: For material/color, examine surface texture and transparency. (재질/색상: 표면 질감 및 투명도 확인)\n"
    "4. Categorization: Apply Korean classification logic (e.g., '비닐' for flexible plastics). (한국식 분류 논리 적용)\n"
    "5. Logic Check: For negative questions ('없는', '아닌', '제외'), verify exclusion criteria. (부정 질문 유의)\n\n"

    "RECYCLING DOMAIN KNOWLEDGE (K-STANDARDS):\n"
    "- 플라스틱 (Plastic): PET bottles, PP/PS containers, rigid plastics.\n"
    "- 유리 (Glass): Glass bottles, jars (transparent/colored).\n"
    "- 금속 (Metal): Aluminum/Steel cans, scrap metal.\n"
    "- 종이 (Paper): Cardboard boxes, paper bags, beverage cartons(우유팩).\n"
    "- 비닐 (Vinyl/Film): Thin, flexible plastic wraps, bags, and pouches.\n"
    "- 스티로폼 (Styrofoam): White expanded polystyrene (EPS).\n\n"

    "QUALITY STANDARD:\n"
    "Choose the single most accurate answer supported by direct visual evidence. "
    "If ambiguous, prioritize the most clearly visible object. "
    "시각적 증거에 기반하여 가장 정확한 답 하나만 선택하십시오. 모호할 경우 가장 명확하게 보이는 객체를 우선합니다."
)


# 프롬프트
def build_mc_prompt(question, a, b, c, d):
    return (
        f"Question: {question}\n"
        f"(a) {a}\n(b) {b}\n(c) {c}\n(d) {d}\n\n"
        "Instruction: Select the correct option and output exactly ONE lowercase letter (a, b, c, or d) only.\n"
        "지시사항: 정답을 선택하고 반드시 소문자 한 글자(a, b, c, d)로만 답변하십시오. 설명은 제외하십시오."
    )

## 선별 샘플 재추론
앞에서 뽑은 어려운/모호한 문제만 순회하며 이미지를 열고 prompt를 만든 뒤 Thinking 추론을 수행합니다.
결과는 로그 텍스트 형태로 남기고, 다음 단계에서 정답만 다시 파싱합니다.

In [13]:
for id in tqdm(ids):
    print(id)

    row = df_test[df_test.id == id].iloc[0]
    img = Image.open(row["path"]).convert("RGB")

    q = str(row["question"])
    a, b, c, d = str(row["a"]), str(row["b"]), str(row["c"]), str(row["d"])
    user_text = build_mc_prompt(q, a, b, c, d)

    messages = [
        {"role":"system","content":[{"type":"text","text":SYSTEM_INSTRUCT}]},
        {"role":"user","content":[
            {"type":"image","image":img},
            {"type":"text","text":user_text}
        ]}
    ]
    # Preparation for inference
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt"
    )
    inputs = inputs.to(model.device)
    # Inference: Generation of the output
    generated_ids = model.generate(**inputs, max_new_tokens=2048)
    generated_ids_trimmed = [
        out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]
    output_text = tokenizer.batch_decode(
        generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )
    print(output_text)

    with open("thinking.txt", "a") as f:
        f.write(id + '\n')
        f.write(str(output_text) + '\n')


  0%|          | 0/226 [00:00<?, ?it/s]

test_1194.jpg
['So, let\'s analyze the image. The question is about how many paper packaging items are visible. First, recall that in Korean recycling standards, "종이" (paper) includes cardboard boxes, paper bags, etc.\n\nLooking at the image:\n\n1. The light green box with "pitem" and product images: this looks like a cardboard box (paper-based).\n2. The small rectangular box with "Coupang" and a barcode: this is also a cardboard box (paper packaging).\n3. The long, narrow item with a green flower: that\'s a hand cream tube, which is probably plastic (since it\'s a rigid tube, but maybe not paper). Wait, the question is about paper. Let\'s check.\n\nWait, the items:\n\n- The large green box (pitem): cardboard, so paper.\n- The Coupang box: cardboard, so paper.\n- The small green and white strip (hand cream): that\'s a plastic tube, not paper.\n- The dark circular object (top right): maybe a cap, but not relevant.\n\nSo how many paper items? The two boxes: the pitem box and the Coupang 

## 1.3 Thinking 모델 추론 결과 파싱

추론 결과에서 정답 파싱하기

## Thinking 로그 읽기
추론 결과를 파일에서 다시 읽어 후처리합니다.
이 흐름은 긴 생성 결과를 노트북 메모리에서 바로 다루기보다 텍스트 로그를 기준으로 안정적으로 파싱하려는 방식입니다.

In [14]:
with open("thinking.txt", "r") as f:
    data = f.readlines()

## 샘플별 출력 분리
전체 로그를 test id 기준으로 잘라 각 문제의 생성 결과를 분리합니다.
이후 셀에서 `</think>` 뒤 최종 선택지 문구만 뽑습니다.

In [15]:
import re
joined = '\n'.join(data)
parsed = re.split(r"(test_\d\d\d\d)", joined)[1:]
ids = parsed[::2]
outs = parsed[1::2]

In [16]:
for id, out in zip(ids, outs):
    print(f"{id}: {out.strip()[-15:]}")

test_4189: </think>\n\nb']
test_0246: </think>\n\nd"]
test_3836: </think>\n\nc"]
test_4189: </think>\n\nb"]
test_1768: </think>\n\nb']
test_2603: </think>\n\nd']
test_1283: </think>\n\nd"]
test_3967: n3. The red (']
test_4606: </think>\n\nd"]
test_4546: </think>\n\na']
test_4343: </think>\n\nc']
test_5051: </think>\n\nb']
test_2937: </think>\n\nc']
test_4970: </think>\n\nb']
test_4825: </think>\n\na']
test_1008: </think>\n\nd"]
test_4490: </think>\n\nc"]
test_0510: </think>\n\na"]
test_1918: </think>\n\na']
test_3487: </think>\n\nc"]
test_0720: </think>\n\nd']
test_4151: </think>\n\nb']
test_2794: </think>\n\na']
test_1063: </think>\n\na']
test_4975: </think>\n\nb"]
test_4481: </think>\n\nd']
test_1194: </think>\n\nc']
test_2587: </think>\n\nc"]
test_1274: </think>\n\nc"]
test_0693:  image again.']
test_4490: </think>\n\nc"]
test_4828: </think>\n\nc']
test_4694: </think>\n\na']
test_1534: </think>\n\nc']
test_1828: </think>\n\nc']
test_1130: </think>\n\nc']
test_1771: </think>\n\nb']
t

## 최종 답안 파싱
모델의 긴 사고 과정 전체를 쓰지 않고, 마지막 응답 구간에서 a/b/c/d 정답만 추출합니다.
형식이 어긋난 출력은 제외 목록으로 남겨 재검토할 수 있게 해둡니다.

In [17]:
# </think> 로 시작하지 않는것들은 제외.

excludes = []
answers = dict()

for id, out in zip(ids, outs):
    end = out.strip()[-15:]
    print(f"{id}: {end}", end="\t")
    if end.startswith("</think>"):
        print(f"answer: {end[-3]}", end="")
        answers[id] = end[-3]
    else:
        excludes.append(id)
    print()

test_4189: </think>\n\nb']	answer: b
test_0246: </think>\n\nd"]	answer: d
test_3836: </think>\n\nc"]	answer: c
test_4189: </think>\n\nb"]	answer: b
test_1768: </think>\n\nb']	answer: b
test_2603: </think>\n\nd']	answer: d
test_1283: </think>\n\nd"]	answer: d
test_3967: n3. The red (']	
test_4606: </think>\n\nd"]	answer: d
test_4546: </think>\n\na']	answer: a
test_4343: </think>\n\nc']	answer: c
test_5051: </think>\n\nb']	answer: b
test_2937: </think>\n\nc']	answer: c
test_4970: </think>\n\nb']	answer: b
test_4825: </think>\n\na']	answer: a
test_1008: </think>\n\nd"]	answer: d
test_4490: </think>\n\nc"]	answer: c
test_0510: </think>\n\na"]	answer: a
test_1918: </think>\n\na']	answer: a
test_3487: </think>\n\nc"]	answer: c
test_0720: </think>\n\nd']	answer: d
test_4151: </think>\n\nb']	answer: b
test_2794: </think>\n\na']	answer: a
test_1063: </think>\n\na']	answer: a
test_4975: </think>\n\nb"]	answer: b
test_4481: </think>\n\nd']	answer: d
test_1194: </think>\n\nc']	answer: c
test_2587:

In [18]:
# print(f"정답을 찾을 수 없는 문제 개수: {len(rethinks)}")

NameError: name 'rethinks' is not defined

In [19]:
df_think = pd.DataFrame({
    "id": answers.keys(),
    "answer": answers.values()
})
df_think.head()

,id,answer
0,test_4189,d
1,test_0246,d
2,test_3836,b
3,test_1768,b
4,test_2603,d


## 기존 submission 덮어쓰기
Thinking 모델이 다시 푼 문제만 기존 주 제출 파일의 answer를 교체합니다.
합의가 높았던 문제는 유지하고, 불확실 문제만 보정하는 후처리 단계입니다.

In [20]:
# Thinking으로 추론한 정답들로 대체
for row in df_think.itertuples():
    main_sub.loc[main_sub.id == row.id, "answer"] = row.answer

## 최종 제출 파일 저장
치환이 끝난 결과를 새로운 submission CSV로 저장합니다.
파일명이 바뀌므로 기존 제출본과 비교하기 쉽습니다.

In [21]:
# 최종 제출 파일 생성
main_sub.to_csv('main-sub-think-2.csv', index=False)

In [ ]:
# 모든 작업 완료 후
print("Job Done! Terminating session in 30 seconds...")
import time
from google.colab import runtime

time.sleep(30)
runtime.unassign()

time.sleep(10)
os.kill(os.getpid(), 9)

Job Done! Terminating session in 30 seconds...


KeyboardInterrupt: 